In [20]:
import tensorflow as tf 
from tensorflow.keras.models import load_model 
import pickle
import numpy as np
import pandas as pd

In [21]:
### Load the trained model, scaler, pickle,onehot
model = load_model('model.h5')

## load the encoder and scaler 
with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)



In [22]:
## Example input data 
input_data = {
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':5000
}

In [23]:
## One-hot encode 'Geography'
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out([ 'Geography' ]))

d:\deeplearningprojects\annclassification\dpann-env\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [24]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [25]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,5000


In [26]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [27]:
## Encode categorical variable
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

In [28]:
## Combine one-hot encoded columns with input data
input_df = pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)

In [29]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,5000,1.0,0.0,0.0


In [30]:
# Scalling the data
input_scaled = scaler.transform(input_df)

In [31]:
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -1.65923237,  1.00150113,
        -0.57946723, -0.57638802]])

In [37]:
### Predict churn 
prediction = model.predict(input_scaled)
prediction_proba = prediction[0][0]
prediction_proba

1/1 [==============================] - 0s 16ms/step


0.036311485

In [38]:
if prediction_proba > 0.5:
    print("The customer is likely to churn")
else:
    print('The customer is not likely to churn')

The customer is not likely to churn
